# Walmart Sales and Weather - Exploratory Data Analysis

This notebook documents the EDA and data-quality work for the ML proof of concept. The goal is to forecast daily item-level unit sales by store using historical sales, calendar information, and local weather observations.

## Project framing

- Business problem: short-term retail demand forecasting.
- Target: `units`, the number of units sold for one `store_nbr`, `item_nbr`, and `date`.
- Input tables: sales history, store-to-weather-station mapping, and daily weather observations.
- Evaluation design: chronological split, because a demand forecasting model must not train on future dates.
- Main modeling risk: the target is extremely sparse, with most rows equal to zero.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

ROOT = Path("..").resolve()
RAW = ROOT / "data" / "raw" / "walmart-recruiting-sales-in-stormy-weather"
PROCESSED = ROOT / "data" / "processed" / "modeling_dataset.parquet"
RESULTS = ROOT / "results"

train_path = RAW / "train.csv"
weather_path = RAW / "weather.csv"
key_path = RAW / "key.csv"

train_path, weather_path, key_path

## Raw data loading

The training file is large but narrow. Explicit dtypes keep memory usage controlled and make repeated exploration faster.

In [ ]:
train = pd.read_csv(
    train_path,
    parse_dates=["date"],
    dtype={"store_nbr": "int16", "item_nbr": "int16", "units": "int16"},
)
key = pd.read_csv(key_path, dtype={"store_nbr": "int16", "station_nbr": "int16"})
weather = pd.read_csv(weather_path)
weather.columns = [column.strip().replace('"', '') for column in weather.columns]

summary = pd.DataFrame(
    {
        "table": ["train", "key", "weather"],
        "rows": [len(train), len(key), len(weather)],
        "columns": [train.shape[1], key.shape[1], weather.shape[1]],
    }
)
summary

In [ ]:
pd.DataFrame(
    [
        {"metric": "stores", "value": train["store_nbr"].nunique()},
        {"metric": "items", "value": train["item_nbr"].nunique()},
        {"metric": "sales_dates", "value": train["date"].nunique()},
        {"metric": "sales_date_min", "value": train["date"].min().date()},
        {"metric": "sales_date_max", "value": train["date"].max().date()},
        {"metric": "weather_stations", "value": weather["station_nbr"].nunique()},
        {"metric": "weather_dates", "value": pd.to_datetime(weather["date"]).nunique()},
    ]
)

## Target distribution

The target is highly zero-inflated: only about 2.57% of rows have positive unit sales. This changes the modeling strategy. A naive model can achieve low average error by predicting values close to zero, so the evaluation must include both global regression metrics and separate error on positive-sales rows.

In [ ]:
target_summary = pd.DataFrame(
    [
        {"metric": "rows", "value": len(train)},
        {"metric": "zero_sales_rows", "value": int((train["units"] == 0).sum())},
        {"metric": "positive_sales_rows", "value": int((train["units"] > 0).sum())},
        {"metric": "zero_sales_share", "value": float((train["units"] == 0).mean())},
        {"metric": "mean_units", "value": float(train["units"].mean())},
        {"metric": "max_units", "value": int(train["units"].max())},
    ]
)
target_summary

In [ ]:
positive_units = train.loc[train["units"] > 0, "units"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(train["units"], bins=80, ax=axes[0])
axes[0].set_title("All unit sales, dominated by zeros")
axes[0].set_xlim(0, 50)

sns.histplot(positive_units, bins=80, ax=axes[1])
axes[1].set_title("Positive sales rows only")
axes[1].set_xlim(0, positive_units.quantile(0.99))
plt.tight_layout()

## Sales concentration

Demand is concentrated in a small set of items and stores. This supports adding store, item, lag, and rolling features, and it also justifies reporting positive-sales error separately from zero-sales error.

In [ ]:
top_items = (
    train.groupby("item_nbr", as_index=False)["units"]
    .sum()
    .sort_values("units", ascending=False)
    .head(15)
)
top_stores = (
    train.groupby("store_nbr", as_index=False)["units"]
    .sum()
    .sort_values("units", ascending=False)
    .head(15)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=top_items, x="item_nbr", y="units", color="#3A6EA5", ax=axes[0])
axes[0].set_title("Top items by total units")
sns.barplot(data=top_stores, x="store_nbr", y="units", color="#3A6EA5", ax=axes[1])
axes[1].set_title("Top stores by total units")
plt.tight_layout()

top_items.head(), top_stores.head()

## Weather data quality

NOAA weather fields contain coded values. `M` means missing, `-` is unavailable for selected fields, and `T` means trace precipitation or snowfall. Trace values are converted to `0.005` inches in the pipeline. Other missing numeric values are imputed by station median first, then global median as a fallback.

In [ ]:
missing_tokens = {"", "M", "-", "T"}
weather_missingness = []
for column in weather.columns:
    values = weather[column].astype(str).str.strip()
    missing_or_coded = values.isin(missing_tokens)
    weather_missingness.append(
        {
            "column": column,
            "missing_or_coded_rows": int(missing_or_coded.sum()),
            "missing_or_coded_share": float(missing_or_coded.mean()),
        }
    )

weather_missingness = pd.DataFrame(weather_missingness).sort_values(
    "missing_or_coded_share", ascending=False
)
weather_missingness.head(12)

In [ ]:
plt.figure(figsize=(11, 4))
sns.barplot(
    data=weather_missingness.head(12),
    x="column",
    y="missing_or_coded_share",
    color="#B55D4C",
)
plt.xticks(rotation=45, ha="right")
plt.title("Weather missing or coded value share")
plt.tight_layout()

## Processed modeling dataset

The processed dataset is produced by `python scripts/prepare_data.py`. It merges sales with store-station mapping and cleaned weather data, then adds date and lag features. Lag and rolling features are shifted, so each row only uses prior sales for the same store-item pair.

In [ ]:
processed = pd.read_parquet(PROCESSED)
processed_summary = pd.DataFrame(
    [
        {"metric": "rows", "value": len(processed)},
        {"metric": "columns", "value": processed.shape[1]},
        {"metric": "missing_feature_cells", "value": int(processed.drop(columns=["date", "units"]).isna().sum().sum())},
        {"metric": "date_min", "value": processed["date"].min().date()},
        {"metric": "date_max", "value": processed["date"].max().date()},
    ]
)
processed_summary

In [ ]:
processed[[
    "date",
    "store_nbr",
    "item_nbr",
    "units",
    "station_nbr",
    "month",
    "dayofweek",
    "lag_7",
    "rolling_mean_28",
    "preciptotal",
    "has_rain",
]].head()

## Chronological validation split

A random split would leak future seasonality and demand patterns into training. The project uses all rows before `2014-07-01` for training and rows from `2014-07-01` onward for testing.

In [ ]:
test_start = pd.Timestamp("2014-07-01")
split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": int((processed["date"] < test_start).sum()),
            "date_min": processed.loc[processed["date"] < test_start, "date"].min().date(),
            "date_max": processed.loc[processed["date"] < test_start, "date"].max().date(),
            "zero_sales_share": float((processed.loc[processed["date"] < test_start, "units"] == 0).mean()),
        },
        {
            "split": "test",
            "rows": int((processed["date"] >= test_start).sum()),
            "date_min": processed.loc[processed["date"] >= test_start, "date"].min().date(),
            "date_max": processed.loc[processed["date"] >= test_start, "date"].max().date(),
            "zero_sales_share": float((processed.loc[processed["date"] >= test_start, "units"] == 0).mean()),
        },
    ]
)
split_summary

## Feature engineering decisions

- Keep store, item, station, and selected calendar variables as categorical features.
- Use numeric calendar variables for trend and within-month position.
- Use weather measurements after cleaning and station-level imputation.
- Convert `codesum` into interpretable binary event flags: rain, snow, fog, thunder, and freezing.
- Add `lag_1`, `lag_7`, `lag_28`, and rolling means over 7, 28, and 90 prior days.
- Use log-transformed targets for ML regressors to reduce the effect of extreme high-unit rows.
- Train ML models on all positive rows plus a controlled zero-row sample, while evaluating on the full chronological test set.

## Modeling plan

The project compares three registered models:

1. Lag blend baseline: transparent benchmark based on previous sales history.
2. Weighted Poisson regression: one-hot encoded categorical features plus scaled numeric features, with weights correcting the zero-row sample.
3. Histogram gradient boosting: non-linear model with deterministic categorical dtypes.

The dashboard is designed for a management audience: it emphasizes model credibility, aggregate realized-versus-predicted demand, product/store sales concentration, and weekly drilldowns for high-volume products.

## EDA conclusions

- There is enough labeled data for a class project without the Kaggle test set.
- The target is extremely sparse, so global metrics alone are not enough.
- Weather needs explicit cleaning before modeling.
- Chronological validation is required to avoid lookahead leakage.
- Lag and rolling store-item features are likely to be the strongest predictors, with weather acting as contextual signal.